# Evidencia de Aprendizaje 1: Base de Datos Analítica sobre Lakehouse
**Asignatura:** Big Data y Procesamiento Distribuido  
**Caso de estudio:** Wanderbricks  
**Estudiante:** Wilfran Camilo Valencia Góez  
**Carrera:** Ingeniería de Software  
**Modalidad:** Individual  
**Repositorio de GitHub:** [https://github.com/06Camilogoez/BigData](https://github.com/06Camilogoez/BigData)  
**Video de Sustentación:** [Pega aquí el enlace de YouTube / Google Drive]

## 1. Exploración de datos, calidad y relaciones

Inspeccionamos el contenido del catálogo `samples.wanderbricks`. El objetivo es revisar el volumen, la calidad (nulos, duplicados), los tipos de datos y las relaciones entre tablas antes de la ingesta.

Trabajamos con seis tablas principales:
- `users` y `properties`: Tablas maestras con datos de cuentas y alojamientos (con coordenadas geográficas).
- `bookings` y `payments`: Datos transaccionales del negocio.
- `clickstream`: Eventos de navegación con estructuras anidadas (structs con detalles de dispositivo y URL).
- `reviews`: Calificaciones y texto libre dejado por los huéspedes.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, DoubleType, StringType, StructField, StructType, TimestampType
from pyspark.sql.window import Window

# Forzamos el catalogo 'samples' activo para que el engine SQL lo resuelva correctamente
# Esto evita el error TABLE_OR_VIEW_NOT_FOUND cuando la sesion tiene otro catalogo activo
spark.sql("USE CATALOG samples")

tablas_fuente = {
    "users":       "samples.wanderbricks.users",
    "properties":  "samples.wanderbricks.properties",
    "bookings":    "samples.wanderbricks.bookings",
    "payments":    "samples.wanderbricks.payments",
    "clickstream": "samples.wanderbricks.clickstream",
    "reviews":     "samples.wanderbricks.reviews",
}

# Usamos spark.sql("SELECT * FROM ...") en lugar de spark.read.table()
# para garantizar que el engine SQL resuelva correctamente las tablas
dfs = {nombre: spark.sql(f"SELECT * FROM {ruta}") for nombre, ruta in tablas_fuente.items()}
print("Tablas cargadas correctamente:", list(dfs.keys()))

# 1. Volumen, duplicados y nulos
metricas = []
for nombre, df in dfs.items():
    total = df.count()
    unicos = df.dropDuplicates().count()
    nulos = sum(df.where(F.col(c).isNull()).count() for c in df.columns)
    metricas.append((nombre, total, unicos, nulos, len(df.columns), str(list(df.columns))))

display(spark.createDataFrame(metricas,
    ["tabla", "total_filas", "filas_distintas", "conteo_nulos", "num_columnas", "columnas"]))

# 2. Calidad del texto libre en reviews
col_texto = next((c for c in ["review_text","review","comment","text","body","description"]
                  if c in dfs["reviews"].columns), None)
print(f"Columna de texto en reviews: {col_texto} | Columnas disponibles: {dfs['reviews'].columns}")
if col_texto:
    display(dfs["reviews"].select(
        F.col(col_texto).alias("comentario"),
        F.length(F.col(col_texto)).alias("caracteres")
    ).agg(
        F.count("*").alias("total_resenas"),
        F.sum(F.when(F.col("comentario").isNull()|(F.trim(F.col("comentario"))==""),1).otherwise(0)).alias("resenas_vacias"),
        F.round(F.avg("caracteres"),2).alias("promedio_caracteres"),
        F.min("caracteres").alias("min_caracteres"),
        F.max("caracteres").alias("max_caracteres")
    ))

# 3. Datos geográficos en properties
col_lat = next((c for c in ["latitude","lat"] if c in dfs["properties"].columns), None)
col_lon = next((c for c in ["longitude","lon","lng"] if c in dfs["properties"].columns), None)
print(f"Columnas geo en properties: lat={col_lat}, lon={col_lon}")
if col_lat and col_lon:
    display(dfs["properties"].agg(
        F.count("*").alias("total_propiedades"),
        F.sum(F.when(F.col(col_lat).isNull()|F.col(col_lon).isNull(),1).otherwise(0)).alias("coordenadas_nulas"),
        F.min(col_lat).alias("lat_min"), F.max(col_lat).alias("lat_max"),
        F.min(col_lon).alias("lon_min"), F.max(col_lon).alias("lon_max")
    ))

# 4. Integridad referencial
def contar_huerfanos(hija, padre, llave):
    if llave in hija.columns and llave in padre.columns:
        return hija.join(padre.select(llave).dropDuplicates(), llave, "left_anti").count()
    return -1

llave_pago = next((c for c in ["booking_id","reservation_id"] if c in dfs["payments"].columns), None)
llave_book = next((c for c in ["booking_id","reservation_id","id"] if c in dfs["bookings"].columns), None)
print(f"Llave primaria bookings: {llave_book} | Llave foranea payments: {llave_pago}")

relaciones = [
    ("bookings -> users",         contar_huerfanos(dfs["bookings"],    dfs["users"],      "user_id")),
    ("bookings -> properties",    contar_huerfanos(dfs["bookings"],    dfs["properties"], "property_id")),
    ("reviews -> properties",     contar_huerfanos(dfs["reviews"],     dfs["properties"], "property_id")),
    ("clickstream -> users",       contar_huerfanos(dfs["clickstream"], dfs["users"],      "user_id")),
]
if llave_pago and llave_book:
    relaciones.append((f"payments -> bookings ({llave_pago})",
                       contar_huerfanos(dfs["payments"], dfs["bookings"], llave_pago)))

display(spark.createDataFrame(relaciones, ["relacion", "registros_huerfanos"]))

print("\nEstructura semiestructurada de clickstream:")
dfs["clickstream"].printSchema()


## 2. Justificación técnica de la arquitectura Lakehouse

Al analizar los datos de Wanderbricks encontramos un escenario mixto: entidades transaccionales estructuradas (`bookings`, `payments`) junto con logs de navegación con structs anidados (`clickstream`) y texto libre (`reviews`). Comparamos tres enfoques:

### Comparativa de paradigmas

| Aspecto evaluado | Relacional (PostgreSQL) | NoSQL (MongoDB) | Lakehouse (Delta Lake) |
| :--- | :--- | :--- | :--- |
| **Logs y JSON anidado** | Obliga a aplanar o usar JSONB con degradación en analítica masiva. | Excelente ingesta sin esquema previo. | Raw crudo en Bronce; desanidado tipado en Plata. |
| **Joins y consultas complejas** | Eficiente para bajo volumen, no escala a millones de eventos. | Joins entre colecciones son lentos y poco naturales. | Joins distribuidos optimizados con Catalyst y Tungsten. |
| **Gobierno y calidad** | ACID estricto pero saturado con telemetría continua. | Consistencia eventual, sin versionado nativo. | ACID completo via `_delta_log`, Time Travel y `mergeSchema`. |
| **Escalabilidad y costo** | Escala vertical, caro para telemetría histórica. | Escala horizontal pero caro en analítica pesada. | Almacenamiento en nube desacoplado del cómputo (S3/ADLS). |

### Decisión: Arquitectura Medallion con Delta Lake
Descartamos el modelo relacional porque `clickstream` y `reviews` saturarían el motor transaccional y cada cambio de tracking requeriría DDLs costosas. Descartamos NoSQL porque el objetivo es analítica OLAP pesada (DAU, embudos, conversión), área donde MongoDB no compite con Spark.

Adoptamos **Medallion**:
1. **Bronce:** Datos crudos inmutables tal como vienen de la fuente.
2. **Plata:** Datos limpios, desanidados, tipados y listos para SQL.
3. **Oro:** Métricas y KPIs de consumo directo para reportes.

## 3. Creación de catálogo, esquemas e ingesta a capa Bronce

Configuramos el catálogo propio `mi_catalogo_ea1` en Unity Catalog con los tres esquemas Medallion y un volumen para archivos crudos. Definimos contratos con `StructType` explícitos e ingerimos todas las tablas a la capa Bronce en formato Delta.

In [ ]:
%sql
-- Creamos el catalogo propio y los tres esquemas de la arquitectura Medallion
CREATE CATALOG IF NOT EXISTS mi_catalogo_ea1;
USE CATALOG mi_catalogo_ea1;

CREATE SCHEMA IF NOT EXISTS bronce;
CREATE SCHEMA IF NOT EXISTS plata;
CREATE SCHEMA IF NOT EXISTS oro;

-- Volumen para archivos crudos externos
CREATE VOLUME IF NOT EXISTS mi_catalogo_ea1.bronce.raw_files;
SHOW VOLUMES IN mi_catalogo_ea1.bronce;

In [ ]:
# Declaramos los contratos de datos formales (StructType explicito)
# para las dos tablas semiestructuradas que requieren tipificacion explicita

print("Schema real de clickstream (fuente):")
dfs["clickstream"].printSchema()

# Construimos el StructType explicito a partir del schema observado
schema_clickstream_explicito = StructType(dfs["clickstream"].schema.fields)
schema_reviews_explicito     = StructType(dfs["reviews"].schema.fields)

print("\nContrato StructType declarado para clickstream:")
print(schema_clickstream_explicito.simpleString())

print("\nContrato StructType declarado para reviews:")
print(schema_reviews_explicito.simpleString())

# Ingerimos todas las tablas a Bronce como Delta (copia inmutable de fuentes)
for nombre, df in dfs.items():
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
      .saveAsTable(f"bronce.{nombre}_raw")
    print(f"  Creada: bronce.{nombre}_raw ({df.count()} filas)")

print("\nTablas en capa Bronce:")
display(spark.sql("SHOW TABLES IN bronce"))

## 4. Limpieza y desanidado para la capa Plata

En Plata dejamos tablas completamente planas para SQL estándar. Si `clickstream` tiene columnas `struct` anidadas, las aplanamos recursivamente extrayendo cada subcampo. También estandarizamos nombres de columnas (`device`, `timestamp`, `event_type`).

In [ ]:
# Función recursiva corregida para aplanar structs sin causar bucles infinitos
def aplanar_columnas(schema, prefijo=""):
    campos = []
    for campo in schema.fields:
        nombre_completo = f"{prefijo}{campo.name}" if prefijo else campo.name
        if isinstance(campo.dataType, StructType):
            campos.extend(aplanar_columnas(campo.dataType, nombre_completo + "."))
        elif isinstance(campo.dataType, ArrayType):
            campos.append(F.explode_outer(F.col(nombre_completo)).alias(campo.name))
        else:
            alias = nombre_completo.replace(".", "_")
            campos.append(F.col(nombre_completo).alias(alias))
    return campos

# Lectura desde Bronce
df_click_bronce     = spark.table("bronce.clickstream_raw")
df_reviews_bronce   = spark.table("bronce.reviews_raw")
df_bookings_bronce  = spark.table("bronce.bookings_raw")
df_props_bronce     = spark.table("bronce.properties_raw")

# Desanidamos clickstream (tabla semiestructurada principal)
df_plata_click = df_click_bronce.select(aplanar_columnas(df_click_bronce.schema))

# Estandarizamos columna 'device' para las consultas
col_device = next((c for c in df_plata_click.columns if "device" in c.lower()), None)
if col_device and col_device != "device":
    df_plata_click = df_plata_click.withColumn("device", F.col(col_device))
elif col_device is None:
    df_plata_click = df_plata_click.withColumn("device", F.lit("unknown"))

# Estandarizamos columna 'event_type' (para resolver el error de event/event_type en SQL)
col_event = next((c for c in df_plata_click.columns if c.lower() in ["event_type", "event", "action", "event_name"]), None)
if col_event and col_event != "event_type":
    df_plata_click = df_plata_click.withColumnRenamed(col_event, "event_type")

# Estandarizamos columna 'timestamp'
col_ts = next((c for c in df_plata_click.columns
               if c.lower() in ["timestamp","event_time","event_timestamp","created_at","ts"]), None)
if col_ts and col_ts != "timestamp":
    df_plata_click = df_plata_click.withColumnRenamed(col_ts, "timestamp")

print("Columnas finales de plata.clickstream:", df_plata_click.columns)

# Desanidamos reviews
df_plata_reviews = df_reviews_bronce.select(aplanar_columnas(df_reviews_bronce.schema))

# Reseteamos físicamente las tablas en DBFS/almacenamiento para limpiar el historial antiguo de Delta
tablas_a_borrar = ["plata.clickstream", "plata.reviews", "plata.bookings", "plata.properties"]
for t in tablas_a_borrar:
    try:
        # Obtenemos la ruta fisica de la tabla
        detalles = spark.sql(f"DESCRIBE DETAIL {t}").collect()[0]
        ruta_fisica = detalles["location"]
        # Dropeamos la tabla de la metadata
        spark.sql(f"DROP TABLE IF EXISTS {t}")
        # Borramos los archivos fisicos para que no herede el log Delta anterior
        dbutils.fs.rm(ruta_fisica, True)
        print(f"Tabla {t} y sus archivos en {ruta_fisica} eliminados correctamente.")
    except Exception as e:
        # Si la tabla no existia previamente
        spark.sql(f"DROP TABLE IF EXISTS {t}")

# Escritura en Plata
df_plata_click.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("plata.clickstream")
df_plata_reviews.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("plata.reviews")
df_bookings_bronce.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("plata.bookings")
df_props_bronce.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("plata.properties")

print("\nVista previa plata.clickstream (desanidada):")
display(spark.table("plata.clickstream").limit(5))

print("Vista previa plata.reviews:")
display(spark.table("plata.reviews").limit(5))


## 5. Demostración práctica de propiedades de Delta Lake

Validamos tres propiedades fundamentales de Delta Lake con operaciones reales sobre `plata.clickstream`:
1. **Atomicidad (ACID):** `UPDATE` transaccional.
2. **Time Travel:** `DESCRIBE HISTORY` y `VERSION AS OF`.
3. **Schema Evolution:** nueva columna con `mergeSchema`.

In [ ]:
%sql
-- 1. ATOMICIDAD: UPDATE transaccional sobre plata.clickstream
-- Delta garantiza que todos los registros afectados se actualizan o ninguno
UPDATE plata.clickstream
SET device = 'desktop_updated'
WHERE device = 'desktop';

SELECT device, COUNT(*) AS total_filas
FROM plata.clickstream
GROUP BY device
ORDER BY total_filas DESC;

In [ ]:
%sql
-- 2. TIME TRAVEL: historial completo de commits de la tabla Delta
DESCRIBE HISTORY plata.clickstream;

In [ ]:
%sql
-- Consulta del estado ANTES del UPDATE usando VERSION AS OF 0
SELECT device, event_type, timestamp
FROM plata.clickstream VERSION AS OF 0
LIMIT 10;

In [ ]:
# 3. SCHEMA EVOLUTION: agregamos 'session_score' con mergeSchema
# Simula el caso real donde producto agrega una métrica sin recrear la tabla
df_con_score = (
    spark.table("plata.clickstream").limit(20)
    .withColumn("session_score",
                F.when(F.col("event_type").isin("booking","reservation","BOOKING"), F.lit(10.0))
                 .when(F.col("event_type").isin("search","SEARCH"), F.lit(5.0))
                 .otherwise(F.lit(1.0)))
)

df_con_score.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable("plata.clickstream")

print("Schema evolucionado (nueva columna session_score):")
display(spark.sql("DESCRIBE plata.clickstream"))

print("\nRegistros con session_score poblado:")
display(
    spark.table("plata.clickstream")
    .filter(F.col("session_score").isNotNull())
    .select("user_id","device","event_type","session_score")
    .limit(5)
)

## 6. Consultas analíticas de negocio

Cinco análisis orientados a decisiones de negocio en Wanderbricks. Cada uno se resuelve en **PySpark** y en **Spark SQL** produciendo el mismo resultado.

### Consulta 1: Embudo de conversión y tasa de reserva por tipo de dispositivo
¿Desde qué dispositivos convierten más los usuarios? Calculamos eventos, usuarios únicos y tasa de conversión a booking.

In [ ]:
# Consulta 1 (PySpark)
clickstream = spark.table("plata.clickstream")

q1_pyspark = (
    clickstream.groupBy("device")
    .agg(
        F.count("*").alias("total_eventos"),
        F.countDistinct("user_id").alias("usuarios_unicos"),
        F.sum(F.when(F.upper(F.col("event_type")).isin("SEARCH"),1).otherwise(0)).alias("busquedas"),
        F.sum(F.when(F.upper(F.col("event_type")).isin("BOOKING","RESERVATION"),1).otherwise(0)).alias("reservas"),
    )
    .withColumn("tasa_reserva", F.round(F.col("reservas")/F.col("total_eventos"),4))
    .orderBy(F.desc("tasa_reserva"), F.desc("usuarios_unicos"))
)
display(q1_pyspark)

In [ ]:
%sql
-- Consulta 1 (SQL)
SELECT
  device,
  COUNT(*) AS total_eventos,
  COUNT(DISTINCT user_id) AS usuarios_unicos,
  SUM(CASE WHEN UPPER(event_type) = 'SEARCH' THEN 1 ELSE 0 END) AS busquedas,
  SUM(CASE WHEN UPPER(event_type) IN ('BOOKING','RESERVATION') THEN 1 ELSE 0 END) AS reservas,
  ROUND(SUM(CASE WHEN UPPER(event_type) IN ('BOOKING','RESERVATION') THEN 1 ELSE 0 END)/COUNT(*),4) AS tasa_reserva
FROM plata.clickstream
GROUP BY device
ORDER BY tasa_reserva DESC, usuarios_unicos DESC;

### Consulta 2: Usuarios Activos Diarios (DAU) y media móvil de 7 días
¿Cómo evoluciona la actividad diaria? Media móvil de 7 días con Window function para suavizar estacionalidad.

In [ ]:
# Consulta 2 (PySpark)
col_ts = next((c for c in clickstream.columns
               if c.lower() in ["timestamp","event_time","event_timestamp","created_at","ts"]), "timestamp")

df_dau = (
    clickstream.withColumn("fecha", F.to_date(col_ts))
    .groupBy("fecha")
    .agg(F.countDistinct("user_id").alias("usuarios_activos"))
)

ventana_7d = Window.orderBy("fecha").rowsBetween(-6, 0)
q2_pyspark = df_dau.withColumn(
    "promedio_movil_7d", F.round(F.avg("usuarios_activos").over(ventana_7d), 2)
).orderBy("fecha")

display(q2_pyspark)

In [ ]:
%sql
-- Consulta 2 (SQL)
WITH usuarios_diarios AS (
  SELECT TO_DATE(timestamp) AS fecha, COUNT(DISTINCT user_id) AS usuarios_activos
  FROM plata.clickstream
  GROUP BY TO_DATE(timestamp)
)
SELECT
  fecha,
  usuarios_activos,
  ROUND(AVG(usuarios_activos) OVER (ORDER BY fecha ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) AS promedio_movil_7d
FROM usuarios_diarios
ORDER BY fecha;

### Consulta 3: Duración aproximada de sesiones por usuario y dispositivo
Estimamos cuánto tiempo pasa cada usuario por sesión calculando la diferencia entre primer y último evento del día.

In [ ]:
# Consulta 3 (PySpark)
q3_pyspark = (
    clickstream.withColumn("fecha_sesion", F.to_date("timestamp"))
    .groupBy("user_id","fecha_sesion","device")
    .agg(
        F.min("timestamp").alias("inicio_sesion"),
        F.max("timestamp").alias("fin_sesion"),
        F.count("*").alias("total_interacciones"),
    )
    .withColumn("duracion_segundos",
                F.col("fin_sesion").cast("long")-F.col("inicio_sesion").cast("long"))
    .filter(F.col("total_interacciones")>=2)
    .orderBy(F.desc("duracion_segundos"))
)
display(q3_pyspark.limit(20))

In [ ]:
%sql
-- Consulta 3 (SQL)
WITH sesiones AS (
  SELECT user_id, TO_DATE(timestamp) AS fecha_sesion, device,
         MIN(timestamp) AS inicio_sesion, MAX(timestamp) AS fin_sesion, COUNT(*) AS total_interacciones
  FROM plata.clickstream
  GROUP BY user_id, TO_DATE(timestamp), device
)
SELECT user_id, fecha_sesion, device, inicio_sesion, fin_sesion, total_interacciones,
       UNIX_TIMESTAMP(fin_sesion)-UNIX_TIMESTAMP(inicio_sesion) AS duracion_segundos
FROM sesiones
WHERE total_interacciones >= 2
ORDER BY duracion_segundos DESC
LIMIT 20;

### Consulta 4: Top eventos por dispositivo con ranking y participación relativa
¿Cuáles son las acciones más frecuentes por dispositivo? Usamos `ROW_NUMBER()` y `SUM() OVER()` para rankear y calcular porcentajes.

In [ ]:
# Consulta 4 (PySpark)
conteo = clickstream.groupBy("device","event_type").agg(F.count("*").alias("conteo"))

w_rank  = Window.partitionBy("device").orderBy(F.desc("conteo"),F.asc("event_type"))
w_total = Window.partitionBy("device")

q4_pyspark = (
    conteo
    .withColumn("ranking",           F.row_number().over(w_rank))
    .withColumn("total_dispositivo", F.sum("conteo").over(w_total))
    .withColumn("participacion",     F.round(F.col("conteo")/F.col("total_dispositivo"),4))
    .filter(F.col("ranking")<=3)
    .orderBy("device","ranking")
)
display(q4_pyspark)

In [ ]:
%sql
-- Consulta 4 (SQL)
WITH eventos AS (
  SELECT device, event_type, COUNT(*) AS conteo
  FROM plata.clickstream GROUP BY device, event_type
), clasificacion AS (
  SELECT device, event_type, conteo,
         ROW_NUMBER() OVER (PARTITION BY device ORDER BY conteo DESC, event_type ASC) AS ranking,
         SUM(conteo) OVER (PARTITION BY device) AS total_dispositivo
  FROM eventos
)
SELECT device, event_type, conteo, ranking, ROUND(conteo/total_dispositivo,4) AS participacion
FROM clasificacion WHERE ranking<=3
ORDER BY device, ranking;

### Consulta 5: Rutas de navegación y matriz de transición de eventos
¿Cuáles son las transiciones más frecuentes entre eventos consecutivos? Usamos `LAG()` para reconstruir la secuencia temporal de cada usuario.

In [ ]:
# Consulta 5 (PySpark)
w_usuario = Window.partitionBy("user_id").orderBy("timestamp")

q5_pyspark = (
    clickstream
    .withColumn("evento_anterior", F.lag("event_type").over(w_usuario))
    .filter(F.col("evento_anterior").isNotNull())
    .groupBy("evento_anterior","event_type")
    .agg(
        F.count("*").alias("total_transiciones"),
        F.countDistinct("user_id").alias("usuarios_distintos"),
    )
    .withColumn("ranking_transicion",
                F.row_number().over(Window.orderBy(F.desc("total_transiciones"),F.asc("evento_anterior"))))
    .orderBy(F.desc("total_transiciones"),F.asc("evento_anterior"))
)
display(q5_pyspark.limit(15))

In [ ]:
%sql
-- Consulta 5 (SQL)
WITH eventos_ordenados AS (
  SELECT user_id, timestamp, event_type,
         LAG(event_type) OVER (PARTITION BY user_id ORDER BY timestamp) AS evento_anterior
  FROM plata.clickstream
), conteo_rutas AS (
  SELECT evento_anterior, event_type,
         COUNT(*) AS total_transiciones, COUNT(DISTINCT user_id) AS usuarios_distintos
  FROM eventos_ordenados
  WHERE evento_anterior IS NOT NULL
  GROUP BY evento_anterior, event_type
)
SELECT evento_anterior, event_type, total_transiciones, usuarios_distintos,
       ROW_NUMBER() OVER (ORDER BY total_transiciones DESC, evento_anterior ASC) AS ranking_transicion
FROM conteo_rutas
ORDER BY total_transiciones DESC, evento_anterior ASC
LIMIT 15;

## 7. Declaración de autoría y herramientas

| Integrante | Usuario GitHub | Aporte y desarrollo | Herramientas de apoyo utilizadas |
| :--- | :--- | :--- | :--- |
| Wilfran Camilo Valencia Góez | [06Camilogoez](https://github.com/06Camilogoez) | Diseño del modelo Lakehouse, scripts de ingesta, desanidado de datos y consultas analíticas (100%). | Databricks Assistant y documentación oficial de Apache Spark / Delta Lake para validación de sintaxis de funciones de ventana. |